# 🔬 Sephora Recommender — Model Seçimi & Hyperparameter Tuning

Bu notebook, öneri sisteminin **ranking katmanı** için farklı modelleri karşılaştırır ve en iyi modeli optimize eder.

## İçerik

| Bölüm | Konu |
|-------|------|
| 1 | Setup & Veri Hazırlığı |
| 2 | Baseline Karşılaştırması (5 model) |
| 3 | Cross-Validation Framework |
| 4 | Optuna ile Hyperparameter Tuning |
| 5 | Final Model Eğitimi |
| 6 | Ensemble Ağırlık Optimizasyonu |
| 7 | Hata Analizi |
| 8 | Model Kaydetme |

## Karşılaştırılan Modeller

```
Ranking Katmanı:
  ├── LightGBM LambdaRank      (pairwise, NDCG optimize)
  ├── XGBoost LambdaMART       (pairwise, güçlü baseline)
  ├── Random Forest Regressor  (pointwise, yorumlanabilir)
  ├── Ridge Regression         (pointwise, hızlı baseline)
  └── CatBoost Ranker          (categorical feature'lar için güçlü)
```

## 0. Kurulum

In [ ]:
# !pip install lightgbm xgboost catboost optuna sentence-transformers scikit-learn pandas numpy matplotlib seaborn shap

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap
import time
import json
import pickle
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

# ML modelleri
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRanker, Pool
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import ndcg_score

# Hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Görsel ayarlar
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Seed
SEED = 42
np.random.seed(SEED)

print('✅ Import başarılı')

## 1. Veri Hazırlığı

Önceki notebook'taki (`sephora_recommendation_model.ipynb`) pipeline'ı aynen kullanıyoruz.

In [ ]:
DATA_DIR   = Path('../data/processed')
MODEL_DIR  = Path('../models')
OUTPUT_DIR = Path('../models/tuning')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rtf = pd.read_parquet(DATA_DIR / 'review_text_features.parquet')
rcl = pd.read_parquet(DATA_DIR / 'review_concern_level.parquet')

print(f'review_text_features : {rtf.shape}')
print(f'review_concern_level : {rcl.shape}')

In [ ]:
# ----------------------------------------------------------------
# Önceki notebook'taki build_aggregate_scores fonksiyonu
# (Buraya kopyalanmış — bağımsız çalışabilmesi için)
# ----------------------------------------------------------------

EFFECT_WEIGHTS = {'helped': 1.0, 'worsened': -1.5, 'target_only': 0.3, 'unknown': 0.0}

def build_aggregate_scores(rcl, rtf):
    df = rcl.copy()
    if 'skin_type' not in df.columns or df['skin_type'].isna().all():
        skin_map = rtf.set_index(['author_id', 'product_id'])['skin_type'].to_dict()
        df['skin_type'] = df.apply(lambda r: skin_map.get((r.get('author_id'), r['product_id']), 'Unknown'), axis=1)
    df['skin_type'] = df['skin_type'].fillna('Unknown')
    df['effect_weight'] = df['effect_label'].map(EFFECT_WEIGHTS).fillna(0.0)
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(3.0)
    df['rating_norm'] = (df['rating'] - 1) / 4
    df['weighted_score'] = df['rating_norm'] * df['effect_weight'] * df['concern_confidence']
    agg = df.groupby(['product_id', 'concern', 'skin_type']).agg(
        product_name=('product_name_final', 'first'),
        brand_name=('brand_name_final', 'first'),
        primary_category=('primary_category', 'first'),
        secondary_category=('secondary_category', 'first'),
        mean_weighted_score=('weighted_score', 'mean'),
        mean_rating=('rating', 'mean'),
        review_count=('product_id', 'count'),
        helped_count=('effect_label', lambda x: (x == 'helped').sum()),
        worsened_count=('effect_label', lambda x: (x == 'worsened').sum()),
        mean_confidence=('concern_confidence', 'mean'),
    ).reset_index()
    agg['review_count_bonus']  = np.log1p(agg['review_count']) / 10
    agg['helped_ratio']        = agg['helped_count'] / agg['review_count'].clip(lower=1)
    agg['worsened_ratio']      = agg['worsened_count'] / agg['review_count'].clip(lower=1)
    agg['aggregate_score']     = agg['mean_weighted_score'] + agg['review_count_bonus']
    return agg

agg_scores = build_aggregate_scores(rcl, rtf)
print(f'Aggregate score tablosu: {agg_scores.shape}')

In [ ]:
def build_ml_features(agg_scores):
    """
    Genişletilmiş feature engineering — tuning notebook'u için ek feature'lar eklendi.
    """
    df = agg_scores.copy()

    # ---- Relevance Label ----
    conditions = [
        (df['worsened_ratio'] > 0.30) | (df['helped_ratio'] < 0.10),
        (df['helped_ratio'] >= 0.10) & (df['helped_ratio'] < 0.35),
        (df['helped_ratio'] >= 0.35) & (df['helped_ratio'] < 0.60),
        (df['helped_ratio'] >= 0.60) & (df['mean_rating'] >= 3.5),
    ]
    df['relevance_label'] = np.select(conditions, [0, 1, 2, 3], default=1)

    # ---- Temel feature'lar ----
    df['log_review_count']    = np.log1p(df['review_count'])
    df['rating_x_helped']     = df['mean_rating'] * df['helped_ratio']
    df['net_effect_ratio']    = df['helped_ratio'] - df['worsened_ratio']
    df['confidence_x_score']  = df['mean_confidence'] * df['mean_weighted_score']
    df['helped_x_confidence'] = df['helped_ratio'] * df['mean_confidence']

    # ---- Ek interaction feature'lar ----
    df['rating_x_net_effect']  = df['mean_rating'] * df['net_effect_ratio']
    df['score_x_log_reviews']  = df['mean_weighted_score'] * df['log_review_count']
    df['helped_sq']            = df['helped_ratio'] ** 2
    df['worsened_sq']          = df['worsened_ratio'] ** 2
    df['review_count_sqrt']    = np.sqrt(df['review_count'])

    # ---- Categorical encode ----
    le_concern  = LabelEncoder()
    le_skin     = LabelEncoder()
    le_pcat     = LabelEncoder()
    le_scat     = LabelEncoder()

    df['concern_enc']    = le_concern.fit_transform(df['concern'].astype(str))
    df['skin_type_enc']  = le_skin.fit_transform(df['skin_type'].astype(str))
    df['pcat_enc']       = le_pcat.fit_transform(df['primary_category'].fillna('Unknown').astype(str))
    df['scat_enc']       = le_scat.fit_transform(df['secondary_category'].fillna('Unknown').astype(str))

    # Query ID
    df['query_id'] = df['concern_enc'].astype(str) + '_' + df['skin_type_enc'].astype(str)
    qle = LabelEncoder()
    df['query_id_enc'] = qle.fit_transform(df['query_id'])

    encoders = {'concern': le_concern, 'skin_type': le_skin, 'pcat': le_pcat, 'scat': le_scat, 'query': qle}
    return df, encoders


ml_df, encoders = build_ml_features(agg_scores)

# Feature grupları
FEATURE_COLS_BASE = [
    'mean_weighted_score', 'mean_rating', 'log_review_count',
    'helped_ratio', 'worsened_ratio', 'net_effect_ratio',
    'mean_confidence', 'review_count_bonus',
    'rating_x_helped', 'confidence_x_score',
    'concern_enc', 'skin_type_enc',
]

FEATURE_COLS_EXTENDED = FEATURE_COLS_BASE + [
    'helped_x_confidence', 'rating_x_net_effect', 'score_x_log_reviews',
    'helped_sq', 'worsened_sq', 'review_count_sqrt',
    'pcat_enc', 'scat_enc',
]

CAT_FEATURES = ['concern_enc', 'skin_type_enc', 'pcat_enc', 'scat_enc']

print(f'Base feature sayısı    : {len(FEATURE_COLS_BASE)}')
print(f'Extended feature sayısı: {len(FEATURE_COLS_EXTENDED)}')
print(f'ML df shape: {ml_df.shape}')
print(f'\nRelevance label dağılımı:')
print(ml_df['relevance_label'].value_counts().sort_index())

## 2. Cross-Validation Framework

**Strateji:** Group-based K-Fold — `query_id` (concern × skin_type) grupları bölünmeden train/val'a ayrılır. Bu, data leakage'ı önler: aynı query'nin bazı ürünleri train'de, bazıları val'da olmaz.

In [ ]:
@dataclass
class CVResult:
    model_name: str
    ndcg5_scores:  List[float] = field(default_factory=list)
    ndcg10_scores: List[float] = field(default_factory=list)
    train_times:   List[float] = field(default_factory=list)

    @property
    def mean_ndcg5(self):  return np.mean(self.ndcg5_scores)
    @property
    def std_ndcg5(self):   return np.std(self.ndcg5_scores)
    @property
    def mean_ndcg10(self): return np.mean(self.ndcg10_scores)
    @property
    def std_ndcg10(self):  return np.std(self.ndcg10_scores)
    @property
    def mean_train_time(self): return np.mean(self.train_times)

    def summary(self) -> Dict:
        return {
            'model': self.model_name,
            'NDCG@5':  f"{self.mean_ndcg5:.4f} ± {self.std_ndcg5:.4f}",
            'NDCG@10': f"{self.mean_ndcg10:.4f} ± {self.std_ndcg10:.4f}",
            'Train Time (s)': f"{self.mean_train_time:.2f}",
        }


def group_kfold_split(df: pd.DataFrame, n_splits: int = 5, seed: int = SEED):
    """
    query_id_enc grubuna göre K-Fold böler.
    Her fold: (train_idx, val_idx) döner.
    """
    unique_queries = df['query_id_enc'].unique()
    np.random.seed(seed)
    np.random.shuffle(unique_queries)

    folds = np.array_split(unique_queries, n_splits)
    splits = []
    for i in range(n_splits):
        val_queries = folds[i]
        train_queries = np.concatenate([folds[j] for j in range(n_splits) if j != i])
        train_idx = df.index[df['query_id_enc'].isin(train_queries)].tolist()
        val_idx   = df.index[df['query_id_enc'].isin(val_queries)].tolist()
        splits.append((train_idx, val_idx))

    return splits


def evaluate_predictions_ndcg(
    df_val: pd.DataFrame,
    pred_scores: np.ndarray,
    k_values: List[int] = [5, 10]
) -> Dict[int, float]:
    """
    Val seti üzerinde grup bazlı NDCG hesaplar.
    """
    df_val = df_val.copy()
    df_val['pred'] = pred_scores

    results = {k: [] for k in k_values}

    for _, grp in df_val.groupby('query_id_enc'):
        if len(grp) < 2:
            continue
        true_rel = grp['relevance_label'].values.reshape(1, -1)
        pred_rel = grp['pred'].values.reshape(1, -1)
        for k in k_values:
            if len(grp) >= k:
                try:
                    results[k].append(ndcg_score(true_rel, pred_rel, k=k))
                except Exception:
                    pass

    return {k: np.mean(v) if v else 0.0 for k, v in results.items()}


# CV splits oluştur
N_SPLITS = 5
cv_splits = group_kfold_split(ml_df, n_splits=N_SPLITS)
print(f'{N_SPLITS}-Fold CV splits hazır.')
for i, (tr, va) in enumerate(cv_splits):
    print(f'  Fold {i+1}: train={len(tr)}, val={len(va)}')

## 3. Baseline Model Karşılaştırması

Her modeli **default hyperparameter**'larla çalıştırıp NDCG@5 ve NDCG@10 karşılaştırıyoruz.

In [ ]:
def get_group_sizes(group_ids: np.ndarray) -> List[int]:
    """Sıralı group array'inden group size listesi döner (LightGBM/XGBoost için)."""
    _, counts = np.unique(group_ids, return_counts=True)
    return counts.tolist()


# ============================================================
# MODEL 1: LightGBM LambdaRank (pairwise)
# ============================================================
def run_lgbm_cv(ml_df, feature_cols, cv_splits, params=None) -> CVResult:
    result = CVResult('LightGBM LambdaRank')
    default_params = {
        'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5, 10],
        'learning_rate': 0.05, 'num_leaves': 31, 'min_data_in_leaf': 5,
        'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
        'label_gain': [0, 1, 3, 7], 'verbose': -1, 'n_jobs': -1,
    }
    if params:
        default_params.update(params)

    for train_idx, val_idx in cv_splits:
        tr = ml_df.loc[train_idx].copy()
        va = ml_df.loc[val_idx].copy()
        X_tr, y_tr, g_tr = tr[feature_cols].values, tr['relevance_label'].values, tr['query_id_enc'].values
        X_va, y_va        = va[feature_cols].values, va['relevance_label'].values

        train_ds = lgb.Dataset(X_tr, label=y_tr, group=get_group_sizes(g_tr))
        t0 = time.time()
        model = lgb.train(default_params, train_ds, num_boost_round=300,
                          callbacks=[lgb.log_evaluation(period=9999)])
        result.train_times.append(time.time() - t0)

        preds = model.predict(X_va)
        ndcg = evaluate_predictions_ndcg(va, preds)
        result.ndcg5_scores.append(ndcg[5])
        result.ndcg10_scores.append(ndcg[10])

    return result


# ============================================================
# MODEL 2: XGBoost LambdaMART (pairwise)
# ============================================================
def run_xgboost_cv(ml_df, feature_cols, cv_splits, params=None) -> CVResult:
    result = CVResult('XGBoost LambdaMART')
    default_params = {
        'objective': 'rank:ndcg', 'eval_metric': 'ndcg@10',
        'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 5,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'n_estimators': 300, 'random_state': SEED,
        'verbosity': 0, 'n_jobs': -1,
    }
    if params:
        default_params.update(params)

    for train_idx, val_idx in cv_splits:
        tr = ml_df.loc[train_idx].copy()
        va = ml_df.loc[val_idx].copy()

        # XGBoost ranker grup sıralamasına ihtiyaç duyar
        tr = tr.sort_values('query_id_enc')
        va = va.sort_values('query_id_enc')

        X_tr, y_tr = tr[feature_cols].values, tr['relevance_label'].values
        X_va        = va[feature_cols].values
        qid_tr = tr['query_id_enc'].values
        qid_va = va['query_id_enc'].values

        model = xgb.XGBRanker(**{k: v for k, v in default_params.items()})
        t0 = time.time()
        model.fit(X_tr, y_tr, qid=qid_tr, verbose=False)
        result.train_times.append(time.time() - t0)

        preds = model.predict(X_va)
        ndcg = evaluate_predictions_ndcg(va, preds)
        result.ndcg5_scores.append(ndcg[5])
        result.ndcg10_scores.append(ndcg[10])

    return result


# ============================================================
# MODEL 3: CatBoost Ranker
# ============================================================
def run_catboost_cv(ml_df, feature_cols, cat_feature_idxs, cv_splits, params=None) -> CVResult:
    result = CVResult('CatBoost Ranker')
    default_params = {
        'loss_function': 'YetiRank',  # NDCG optimize eden CatBoost objective
        'learning_rate': 0.05, 'depth': 6, 'iterations': 300,
        'random_seed': SEED, 'verbose': False,
        'eval_metric': 'NDCG',
    }
    if params:
        default_params.update(params)

    for train_idx, val_idx in cv_splits:
        tr = ml_df.loc[train_idx].sort_values('query_id_enc')
        va = ml_df.loc[val_idx].sort_values('query_id_enc')

        X_tr, y_tr = tr[feature_cols].values, tr['relevance_label'].values
        X_va        = va[feature_cols].values
        g_tr = tr['query_id_enc'].values

        pool_tr = Pool(X_tr, label=y_tr, group_id=g_tr, cat_features=cat_feature_idxs)
        model   = CatBoostRanker(**default_params)

        t0 = time.time()
        model.fit(pool_tr)
        result.train_times.append(time.time() - t0)

        preds = model.predict(X_va)
        ndcg = evaluate_predictions_ndcg(va, preds)
        result.ndcg5_scores.append(ndcg[5])
        result.ndcg10_scores.append(ndcg[10])

    return result


# ============================================================
# MODEL 4: Random Forest (pointwise)
# ============================================================
def run_rf_cv(ml_df, feature_cols, cv_splits, params=None) -> CVResult:
    result = CVResult('Random Forest')
    default_params = {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 5,
                      'n_jobs': -1, 'random_state': SEED}
    if params:
        default_params.update(params)

    for train_idx, val_idx in cv_splits:
        tr = ml_df.loc[train_idx]
        va = ml_df.loc[val_idx]
        X_tr, y_tr = tr[feature_cols].values, tr['relevance_label'].values
        X_va        = va[feature_cols].values

        model = RandomForestRegressor(**default_params)
        t0 = time.time()
        model.fit(X_tr, y_tr)
        result.train_times.append(time.time() - t0)

        preds = model.predict(X_va)
        ndcg = evaluate_predictions_ndcg(va, preds)
        result.ndcg5_scores.append(ndcg[5])
        result.ndcg10_scores.append(ndcg[10])

    return result


# ============================================================
# MODEL 5: Ridge Regression (pointwise, hızlı baseline)
# ============================================================
def run_ridge_cv(ml_df, feature_cols, cv_splits, params=None) -> CVResult:
    result = CVResult('Ridge Regression')
    default_params = {'alpha': 1.0}
    if params:
        default_params.update(params)

    scaler = StandardScaler()

    for train_idx, val_idx in cv_splits:
        tr = ml_df.loc[train_idx]
        va = ml_df.loc[val_idx]
        X_tr = scaler.fit_transform(tr[feature_cols].values)
        X_va = scaler.transform(va[feature_cols].values)
        y_tr = tr['relevance_label'].values

        model = Ridge(**default_params)
        t0 = time.time()
        model.fit(X_tr, y_tr)
        result.train_times.append(time.time() - t0)

        preds = model.predict(X_va)
        ndcg = evaluate_predictions_ndcg(va, preds)
        result.ndcg5_scores.append(ndcg[5])
        result.ndcg10_scores.append(ndcg[10])

    return result


print('Tüm model fonksiyonları tanımlandı.')

In [ ]:
# CatBoost için categorical feature indekslerini bul
cat_feature_idxs = [FEATURE_COLS_EXTENDED.index(f) for f in CAT_FEATURES if f in FEATURE_COLS_EXTENDED]

print('🏁 Baseline karşılaştırması başlıyor...')
print(f'   Feature seti: EXTENDED ({len(FEATURE_COLS_EXTENDED)} feature)')
print(f'   CV: {N_SPLITS}-Fold Group K-Fold')
print()

all_results: Dict[str, CVResult] = {}

print('[ 1/5 ] LightGBM LambdaRank...')
all_results['lgbm'] = run_lgbm_cv(ml_df, FEATURE_COLS_EXTENDED, cv_splits)
print(f"        NDCG@10: {all_results['lgbm'].mean_ndcg10:.4f} ± {all_results['lgbm'].std_ndcg10:.4f}")

print('[ 2/5 ] XGBoost LambdaMART...')
all_results['xgb'] = run_xgboost_cv(ml_df, FEATURE_COLS_EXTENDED, cv_splits)
print(f"        NDCG@10: {all_results['xgb'].mean_ndcg10:.4f} ± {all_results['xgb'].std_ndcg10:.4f}")

print('[ 3/5 ] CatBoost Ranker...')
all_results['catboost'] = run_catboost_cv(ml_df, FEATURE_COLS_EXTENDED, cat_feature_idxs, cv_splits)
print(f"        NDCG@10: {all_results['catboost'].mean_ndcg10:.4f} ± {all_results['catboost'].std_ndcg10:.4f}")

print('[ 4/5 ] Random Forest...')
all_results['rf'] = run_rf_cv(ml_df, FEATURE_COLS_EXTENDED, cv_splits)
print(f"        NDCG@10: {all_results['rf'].mean_ndcg10:.4f} ± {all_results['rf'].std_ndcg10:.4f}")

print('[ 5/5 ] Ridge Regression...')
all_results['ridge'] = run_ridge_cv(ml_df, FEATURE_COLS_EXTENDED, cv_splits)
print(f"        NDCG@10: {all_results['ridge'].mean_ndcg10:.4f} ± {all_results['ridge'].std_ndcg10:.4f}")

print('\n✅ Tüm baseline modeller tamamlandı.')

In [ ]:
# ---- Özet tablo ----
summary_df = pd.DataFrame([r.summary() for r in all_results.values()])
print('\n' + '='*65)
print('BASELINE MODEL KARŞILAŞTIRMASI')
print('='*65)
print(summary_df.to_string(index=False))
print('='*65)

In [ ]:
# ---- Görselleştirme ----
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

model_names = [r.model_name for r in all_results.values()]
ndcg5_means  = [r.mean_ndcg5  for r in all_results.values()]
ndcg5_stds   = [r.std_ndcg5   for r in all_results.values()]
ndcg10_means = [r.mean_ndcg10 for r in all_results.values()]
ndcg10_stds  = [r.std_ndcg10  for r in all_results.values()]
train_times  = [r.mean_train_time for r in all_results.values()]

# Sort by NDCG@10
order = np.argsort(ndcg10_means)[::-1]

# NDCG@5
axes[0].barh(
    [model_names[i] for i in order],
    [ndcg5_means[i] for i in order],
    xerr=[ndcg5_stds[i] for i in order],
    capsize=4, color=COLORS[:len(order)]
)
axes[0].set_title('NDCG@5 (mean ± std)', fontsize=13, fontweight='bold')
axes[0].set_xlim(0, 1)
axes[0].axvline(0.5, color='red', ls='--', alpha=0.4)

# NDCG@10
axes[1].barh(
    [model_names[i] for i in order],
    [ndcg10_means[i] for i in order],
    xerr=[ndcg10_stds[i] for i in order],
    capsize=4, color=COLORS[:len(order)]
)
axes[1].set_title('NDCG@10 (mean ± std)', fontsize=13, fontweight='bold')
axes[1].set_xlim(0, 1)
axes[1].axvline(0.5, color='red', ls='--', alpha=0.4)

# Train time
axes[2].barh(
    [model_names[i] for i in order],
    [train_times[i] for i in order],
    color=COLORS[:len(order)]
)
axes[2].set_title('Ortalama Eğitim Süresi (sn)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Saniye')

plt.suptitle('Baseline Model Karşılaştırması — 5-Fold CV', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi.')

In [ ]:
# Fold bazında NDCG dağılımı (box plot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in [
    (axes[0], 'ndcg5_scores',  'NDCG@5  per Fold'),
    (axes[1], 'ndcg10_scores', 'NDCG@10 per Fold'),
]:
    data = [getattr(r, metric) for r in all_results.values()]
    bp = ax.boxplot(data, labels=model_names, patch_artist=True, vert=True)
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=15)
    ax.axhline(0.5, color='red', ls='--', alpha=0.3)

plt.suptitle('Fold Bazında NDCG Dağılımı', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fold_ndcg_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# En iyi model belirleme
best_model_key = max(all_results.keys(), key=lambda k: all_results[k].mean_ndcg10)
best_result    = all_results[best_model_key]

print(f'\n🏆 En iyi baseline model: {best_result.model_name}')
print(f'   NDCG@5  : {best_result.mean_ndcg5:.4f} ± {best_result.std_ndcg5:.4f}')
print(f'   NDCG@10 : {best_result.mean_ndcg10:.4f} ± {best_result.std_ndcg10:.4f}')
print(f'   Eğitim  : {best_result.mean_train_time:.2f}s/fold')

## 4. Feature Set Karşılaştırması

Base feature set vs Extended feature set — ekstra feature'lar ne kadar katkı sağlıyor?

In [ ]:
print('Feature set karşılaştırması — LightGBM ile...')

lgbm_base = run_lgbm_cv(ml_df, FEATURE_COLS_BASE, cv_splits)
lgbm_base.model_name = 'LightGBM (base features)'

lgbm_ext = run_lgbm_cv(ml_df, FEATURE_COLS_EXTENDED, cv_splits)
lgbm_ext.model_name = 'LightGBM (extended features)'

feat_comp = pd.DataFrame([
    {'Feature Set': 'Base',     'Features': len(FEATURE_COLS_BASE),
     'NDCG@5': lgbm_base.mean_ndcg5,  'NDCG@10': lgbm_base.mean_ndcg10},
    {'Feature Set': 'Extended', 'Features': len(FEATURE_COLS_EXTENDED),
     'NDCG@5': lgbm_ext.mean_ndcg5,  'NDCG@10': lgbm_ext.mean_ndcg10},
])
print(feat_comp.to_string(index=False))

delta_ndcg10 = lgbm_ext.mean_ndcg10 - lgbm_base.mean_ndcg10
print(f'\nExtended → NDCG@10 delta: {delta_ndcg10:+.4f}')

## 5. Optuna ile Hyperparameter Tuning

En iyi performans gösteren modeli Optuna ile optimize ediyoruz.

**Optuna neden?**
- Grid search'e göre çok daha verimli (Tree-structured Parzen Estimator)
- Pruning: kötü trial'ları erken keser
- Paralel çalışabilir
- Her trial'ın detaylı logunu tutar

In [ ]:
# ============================================================
# Optuna: LightGBM LambdaRank
# ============================================================

def lgbm_objective(trial: optuna.Trial) -> float:
    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'ndcg_eval_at': [10],
        'label_gain': [0, 1, 3, 7],
        'verbose': -1,
        'n_jobs': -1,
        # ---- Tuning parametreleri ----
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 15, 127),
        'max_depth':          trial.suggest_int('max_depth', 3, 12),
        'min_data_in_leaf':   trial.suggest_int('min_data_in_leaf', 3, 30),
        'feature_fraction':   trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':   trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':       trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1':          trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2':          trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        'min_gain_to_split':  trial.suggest_float('min_gain_to_split', 0.0, 0.5),
    }
    num_boost_round = trial.suggest_int('num_boost_round', 100, 600)

    # 3-fold CV (daha hızlı tuning)
    fast_splits = group_kfold_split(ml_df, n_splits=3)
    ndcg10_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(fast_splits):
        tr = ml_df.loc[train_idx]
        va = ml_df.loc[val_idx]
        X_tr, y_tr, g_tr = tr[FEATURE_COLS_EXTENDED].values, tr['relevance_label'].values, tr['query_id_enc'].values
        X_va = va[FEATURE_COLS_EXTENDED].values

        ds = lgb.Dataset(X_tr, label=y_tr, group=get_group_sizes(g_tr))
        model = lgb.train(params, ds, num_boost_round=num_boost_round,
                          callbacks=[lgb.log_evaluation(period=9999)])

        preds = model.predict(X_va)
        ndcg = evaluate_predictions_ndcg(va, preds)
        ndcg10_scores.append(ndcg[10])

        # Intermediate pruning
        trial.report(np.mean(ndcg10_scores), fold_idx)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(ndcg10_scores)


print('Optuna LightGBM tuning başlıyor...')
print('(N_TRIALS=50 — daha iyi sonuç için artırın)')

lgbm_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
)

N_TRIALS = 50  # Production için 100-200 öneririz
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ LightGBM tuning tamamlandı')
print(f'   En iyi NDCG@10: {lgbm_study.best_value:.4f}')
print(f'   En iyi parametreler:')
for k, v in lgbm_study.best_params.items():
    print(f'     {k}: {v}')

In [ ]:
# ============================================================
# Optuna: XGBoost LambdaMART
# ============================================================

def xgb_objective(trial: optuna.Trial) -> float:
    params = {
        'objective': 'rank:ndcg',
        'verbosity': 0, 'n_jobs': -1, 'random_state': SEED,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 20),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
    }

    fast_splits = group_kfold_split(ml_df, n_splits=3)
    ndcg10_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(fast_splits):
        tr = ml_df.loc[train_idx].sort_values('query_id_enc')
        va = ml_df.loc[val_idx].sort_values('query_id_enc')

        model = xgb.XGBRanker(**params)
        model.fit(tr[FEATURE_COLS_EXTENDED].values, tr['relevance_label'].values,
                  qid=tr['query_id_enc'].values, verbose=False)

        preds = model.predict(va[FEATURE_COLS_EXTENDED].values)
        ndcg = evaluate_predictions_ndcg(va, preds)
        ndcg10_scores.append(ndcg[10])

        trial.report(np.mean(ndcg10_scores), fold_idx)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(ndcg10_scores)


print('Optuna XGBoost tuning başlıyor...')

xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
)
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ XGBoost tuning tamamlandı')
print(f'   En iyi NDCG@10: {xgb_study.best_value:.4f}')
print(f'   En iyi parametreler: {xgb_study.best_params}')

In [ ]:
# ============================================================
# Optuna: CatBoost Ranker
# ============================================================

def catboost_objective(trial: optuna.Trial) -> float:
    params = {
        'loss_function': 'YetiRank',
        'eval_metric': 'NDCG',
        'random_seed': SEED, 'verbose': False,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':         trial.suggest_int('depth', 3, 10),
        'iterations':    trial.suggest_int('iterations', 100, 500),
        'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'border_count':  trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }

    fast_splits = group_kfold_split(ml_df, n_splits=3)
    ndcg10_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(fast_splits):
        tr = ml_df.loc[train_idx].sort_values('query_id_enc')
        va = ml_df.loc[val_idx].sort_values('query_id_enc')

        pool_tr = Pool(
            tr[FEATURE_COLS_EXTENDED].values,
            label=tr['relevance_label'].values,
            group_id=tr['query_id_enc'].values,
            cat_features=cat_feature_idxs
        )
        model = CatBoostRanker(**params)
        model.fit(pool_tr)

        preds = model.predict(va[FEATURE_COLS_EXTENDED].values)
        ndcg = evaluate_predictions_ndcg(va, preds)
        ndcg10_scores.append(ndcg[10])

        trial.report(np.mean(ndcg10_scores), fold_idx)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(ndcg10_scores)


print('Optuna CatBoost tuning başlıyor...')

cb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
)
cb_study.optimize(catboost_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ CatBoost tuning tamamlandı')
print(f'   En iyi NDCG@10: {cb_study.best_value:.4f}')
print(f'   En iyi parametreler: {cb_study.best_params}')

In [ ]:
# ---- Optuna sonuçlarını görselleştir ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

studies  = [lgbm_study,    xgb_study,    cb_study]
titles   = ['LightGBM',    'XGBoost',    'CatBoost']

for ax, study, title in zip(axes, studies, titles):
    trials_df = study.trials_dataframe()
    trials_df = trials_df[trials_df['state'] == 'COMPLETE']
    ax.plot(trials_df['number'], trials_df['value'], alpha=0.4, color='steelblue', linewidth=0.8)
    # Running best
    running_best = trials_df['value'].cummax()
    ax.plot(trials_df['number'], running_best, color='crimson', linewidth=2, label='Running Best')
    ax.set_title(f'{title} — Optuna Trial History', fontsize=12, fontweight='bold')
    ax.set_xlabel('Trial')
    ax.set_ylabel('NDCG@10')
    ax.legend()
    ax.annotate(f'Best: {study.best_value:.4f}',
                xy=(0.02, 0.92), xycoords='axes fraction', fontsize=10,
                bbox=dict(boxstyle='round', fc='yellow', alpha=0.5))

plt.suptitle('Optuna Hyperparameter Search History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Parametre önem analizi (Optuna)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, study, title in zip(axes, studies, titles):
    try:
        importances = optuna.importance.get_param_importances(study)
        imp_df = pd.DataFrame(list(importances.items()), columns=['param', 'importance'])
        imp_df = imp_df.sort_values('importance', ascending=True).tail(10)
        ax.barh(imp_df['param'], imp_df['importance'])
        ax.set_title(f'{title} — Param Importance', fontsize=12, fontweight='bold')
        ax.set_xlabel('Importance')
    except Exception as e:
        ax.text(0.5, 0.5, f'Hesaplanamadı:\n{e}', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)

plt.suptitle('Hyperparameter Önem Analizi (Optuna FAnova)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'param_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Tuned Modellerin 5-Fold CV Karşılaştırması

In [ ]:
print('Tuned modeller 5-Fold CV ile değerlendiriliyor...')

tuned_results = {}

# LightGBM tuned
print('[ 1/3 ] LightGBM (tuned)...')
lgbm_tuned_params = {k: v for k, v in lgbm_study.best_params.items() if k != 'num_boost_round'}
lgbm_tuned_params['num_boost_round'] = lgbm_study.best_params.get('num_boost_round', 300)
r = run_lgbm_cv(ml_df, FEATURE_COLS_EXTENDED, cv_splits, params=lgbm_tuned_params)
r.model_name = 'LightGBM (tuned)'
tuned_results['lgbm_tuned'] = r
print(f'   NDCG@10: {r.mean_ndcg10:.4f} ± {r.std_ndcg10:.4f}')

# XGBoost tuned
print('[ 2/3 ] XGBoost (tuned)...')
r = run_xgboost_cv(ml_df, FEATURE_COLS_EXTENDED, cv_splits, params=xgb_study.best_params)
r.model_name = 'XGBoost (tuned)'
tuned_results['xgb_tuned'] = r
print(f'   NDCG@10: {r.mean_ndcg10:.4f} ± {r.std_ndcg10:.4f}')

# CatBoost tuned
print('[ 3/3 ] CatBoost (tuned)...')
r = run_catboost_cv(ml_df, FEATURE_COLS_EXTENDED, cat_feature_idxs, cv_splits, params=cb_study.best_params)
r.model_name = 'CatBoost (tuned)'
tuned_results['cb_tuned'] = r
print(f'   NDCG@10: {r.mean_ndcg10:.4f} ± {r.std_ndcg10:.4f}')

In [ ]:
# Tüm sonuçlar: baseline + tuned karşılaştırma tablosu
combined = {**all_results, **tuned_results}

comp_table = pd.DataFrame([
    {
        'Model': r.model_name,
        'NDCG@5 (mean)':  round(r.mean_ndcg5, 4),
        'NDCG@5 (std)':   round(r.std_ndcg5, 4),
        'NDCG@10 (mean)': round(r.mean_ndcg10, 4),
        'NDCG@10 (std)':  round(r.std_ndcg10, 4),
        'Train Time (s)': round(r.mean_train_time, 2),
        'Type': 'Tuned' if 'tuned' in k else 'Baseline'
    }
    for k, r in combined.items()
]).sort_values('NDCG@10 (mean)', ascending=False)

print('\n' + '='*90)
print('TÜM MODEL KARŞILAŞTIRMASI (Baseline + Tuned)')
print('='*90)
print(comp_table.to_string(index=False))
print('='*90)

comp_table.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
print('CSV kaydedildi.')

In [ ]:
# Baseline vs Tuned gain görselleştirmesi
pairs = [
    ('lgbm', 'lgbm_tuned', 'LightGBM'),
    ('xgb',  'xgb_tuned',  'XGBoost'),
    ('catboost', 'cb_tuned', 'CatBoost'),
]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(pairs))
width = 0.35

base_scores  = [all_results[b].mean_ndcg10 for b, t, n in pairs]
tuned_scores = [tuned_results[t].mean_ndcg10 for b, t, n in pairs]
names        = [n for b, t, n in pairs]

bars1 = ax.bar(x - width/2, base_scores,  width, label='Baseline', alpha=0.8, color='steelblue')
bars2 = ax.bar(x + width/2, tuned_scores, width, label='Tuned',    alpha=0.8, color='crimson')

for bar, score in zip(bars1, base_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.4f}', ha='center', va='bottom', fontsize=9)
for bar, score in zip(bars2, tuned_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.4f}', ha='center', va='bottom', fontsize=9, color='crimson')

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=12)
ax.set_ylabel('NDCG@10')
ax.set_title('Baseline vs Tuned — NDCG@10 Karşılaştırması', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, max(tuned_scores) * 1.15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'baseline_vs_tuned.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Final Model Eğitimi

En iyi tuned model tüm veri üzerinde eğitilir.

In [ ]:
# En iyi tuned modeli belirle
best_tuned_key  = max(tuned_results.keys(), key=lambda k: tuned_results[k].mean_ndcg10)
best_tuned_name = tuned_results[best_tuned_key].model_name
print(f'🏆 Final model: {best_tuned_name}')
print(f'   NDCG@10: {tuned_results[best_tuned_key].mean_ndcg10:.4f}')

# Tüm veriyle eğit (final model)
X_all = ml_df[FEATURE_COLS_EXTENDED].values
y_all = ml_df['relevance_label'].values
g_all = ml_df['query_id_enc'].values

if best_tuned_key == 'lgbm_tuned':
    final_params = {k: v for k, v in lgbm_study.best_params.items() if k != 'num_boost_round'}
    final_params.update({
        'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5, 10],
        'label_gain': [0, 1, 3, 7], 'verbose': -1, 'n_jobs': -1
    })
    n_rounds = lgbm_study.best_params.get('num_boost_round', 300)
    train_ds = lgb.Dataset(X_all, label=y_all, group=get_group_sizes(g_all))
    final_model = lgb.train(final_params, train_ds, num_boost_round=n_rounds,
                            callbacks=[lgb.log_evaluation(period=9999)])
    final_model_type = 'lgbm'

elif best_tuned_key == 'xgb_tuned':
    final_params = {**xgb_study.best_params, 'verbosity': 0, 'n_jobs': -1, 'random_state': SEED,
                    'objective': 'rank:ndcg'}
    ml_df_sorted = ml_df.sort_values('query_id_enc')
    final_model = xgb.XGBRanker(**final_params)
    final_model.fit(ml_df_sorted[FEATURE_COLS_EXTENDED].values,
                    ml_df_sorted['relevance_label'].values,
                    qid=ml_df_sorted['query_id_enc'].values, verbose=False)
    final_model_type = 'xgb'

else:  # catboost
    final_params = {**cb_study.best_params, 'loss_function': 'YetiRank',
                    'eval_metric': 'NDCG', 'random_seed': SEED, 'verbose': False}
    ml_df_sorted = ml_df.sort_values('query_id_enc')
    pool_all = Pool(
        ml_df_sorted[FEATURE_COLS_EXTENDED].values,
        label=ml_df_sorted['relevance_label'].values,
        group_id=ml_df_sorted['query_id_enc'].values,
        cat_features=cat_feature_idxs
    )
    final_model = CatBoostRanker(**final_params)
    final_model.fit(pool_all)
    final_model_type = 'catboost'

print(f'\n✅ Final model eğitimi tamamlandı: {best_tuned_name}')

In [ ]:
# Final model feature importance
fig, ax = plt.subplots(figsize=(10, 7))

if final_model_type == 'lgbm':
    importance = final_model.feature_importance(importance_type='gain')
    feat_names = FEATURE_COLS_EXTENDED
elif final_model_type == 'xgb':
    importance = final_model.feature_importances_
    feat_names = FEATURE_COLS_EXTENDED
else:
    importance = final_model.get_feature_importance()
    feat_names = FEATURE_COLS_EXTENDED

imp_df = pd.DataFrame({'feature': feat_names, 'importance': importance})
imp_df = imp_df.sort_values('importance', ascending=True)

colors = ['crimson' if i >= len(imp_df) - 5 else 'steelblue' for i in range(len(imp_df))]
ax.barh(imp_df['feature'], imp_df['importance'], color=colors)
ax.set_title(f'Feature Importance — {best_tuned_name}', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance (Gain)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'final_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP analizi (LightGBM/XGBoost için)
if final_model_type in ('lgbm', 'xgb'):
    print('SHAP değerleri hesaplanıyor...')

    # Küçük bir subset üzerinde hesapla (hız için)
    sample_size = min(500, len(ml_df))
    X_sample = ml_df[FEATURE_COLS_EXTENDED].sample(sample_size, random_state=SEED).values

    if final_model_type == 'lgbm':
        explainer  = shap.TreeExplainer(final_model)
    else:
        explainer  = shap.TreeExplainer(final_model)

    shap_values = explainer.shap_values(X_sample)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    plt.sca(axes[0])
    shap.summary_plot(
        shap_values, X_sample,
        feature_names=FEATURE_COLS_EXTENDED,
        plot_type='bar', show=False
    )
    axes[0].set_title('SHAP Feature Importance (Bar)', fontsize=12)

    plt.sca(axes[1])
    shap.summary_plot(
        shap_values, X_sample,
        feature_names=FEATURE_COLS_EXTENDED,
        show=False
    )
    axes[1].set_title('SHAP Beeswarm — Feature Etki Dağılımı', fontsize=12)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'shap_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('SHAP analizi tamamlandı.')
else:
    print('SHAP analizi LightGBM/XGBoost için yapılır. CatBoost built-in importance kullanılıyor.')

## 8. Ensemble Ağırlık Optimizasyonu (Optuna)

Aggregate score, final model score ve semantic score'un ağırlıklarını optimize ediyoruz.

In [ ]:
# Final model skorlarını ml_df'e ekle
if final_model_type == 'lgbm':
    ml_df['final_model_score'] = final_model.predict(ml_df[FEATURE_COLS_EXTENDED].values)
elif final_model_type == 'xgb':
    ml_df['final_model_score'] = final_model.predict(ml_df[FEATURE_COLS_EXTENDED].values)
else:
    ml_df['final_model_score'] = final_model.predict(ml_df[FEATURE_COLS_EXTENDED].values)

# Group içinde normalize
def group_minmax(df, col):
    return df.groupby('query_id_enc')[col].transform(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9))

ml_df['agg_norm']   = group_minmax(ml_df, 'aggregate_score')
ml_df['model_norm'] = group_minmax(ml_df, 'final_model_score')

# Semantic skorları hesapla (önceki notebook'tan yükleniyorsa atla)
try:
    with open(MODEL_DIR / 'product_concern_embeddings.pkl', 'rb') as f:
        product_concern_embeddings = pickle.load(f)
    from sentence_transformers import SentenceTransformer
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    HAS_SEMANTIC = True
    print('✅ Semantic embeddings yüklendi')
except FileNotFoundError:
    HAS_SEMANTIC = False
    print('⚠️ Semantic embeddings bulunamadı — 2-way ensemble optimize edilecek')

In [ ]:
def ensemble_weight_objective(trial: optuna.Trial) -> float:
    """
    Ensemble ağırlıklarını optimize eder.
    Semantic yok ise 2-way (agg + model), varsa 3-way optimize eder.
    """
    w_agg   = trial.suggest_float('w_agg', 0.0, 1.0)
    w_model = trial.suggest_float('w_model', 0.0, 1.0 - w_agg)
    w_sem   = 1.0 - w_agg - w_model  # kalan ağırlık semantic'e

    if w_sem < 0:
        return 0.0

    if HAS_SEMANTIC:
        ens = w_agg * ml_df['agg_norm'] + w_model * ml_df['model_norm'] + w_sem * ml_df.get('sem_norm', 0)
    else:
        w_total = w_agg + w_model
        if w_total == 0: return 0.0
        ens = (w_agg * ml_df['agg_norm'] + w_model * ml_df['model_norm']) / w_total

    ml_df_temp = ml_df.copy()
    ml_df_temp['ens'] = ens

    ndcg10_scores = []
    for _, grp in ml_df_temp.groupby('query_id_enc'):
        if len(grp) >= 5:
            try:
                n = ndcg_score(
                    grp['relevance_label'].values.reshape(1, -1),
                    grp['ens'].values.reshape(1, -1),
                    k=10
                )
                ndcg10_scores.append(n)
            except Exception:
                pass

    return np.mean(ndcg10_scores) if ndcg10_scores else 0.0


print('Ensemble ağırlık optimizasyonu başlıyor...')

weight_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
weight_study.optimize(ensemble_weight_objective, n_trials=200, show_progress_bar=True)

best_w = weight_study.best_params
w_agg_opt   = best_w['w_agg']
w_model_opt = best_w['w_model']
w_sem_opt   = max(0, 1.0 - w_agg_opt - w_model_opt)

print(f'\n✅ Ensemble ağırlık optimizasyonu tamamlandı')
print(f'   w_aggregate : {w_agg_opt:.4f}')
print(f'   w_model     : {w_model_opt:.4f}')
print(f'   w_semantic  : {w_sem_opt:.4f}')
print(f'   NDCG@10     : {weight_study.best_value:.4f}')

In [ ]:
# Ağırlık optimizasyonu görselleştirmesi
trials_df = weight_study.trials_dataframe()
trials_df = trials_df[trials_df['state'] == 'COMPLETE']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial history
axes[0].plot(trials_df['number'], trials_df['value'], alpha=0.3, color='steelblue')
axes[0].plot(trials_df['number'], trials_df['value'].cummax(), color='crimson', lw=2, label='Running Best')
axes[0].set_title('Ensemble Weight Optimization History', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('NDCG@10')
axes[0].legend()
axes[0].annotate(f'Best: {weight_study.best_value:.4f}\nw_agg={w_agg_opt:.3f}, w_model={w_model_opt:.3f}, w_sem={w_sem_opt:.3f}',
                 xy=(0.02, 0.85), xycoords='axes fraction', fontsize=9,
                 bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8))

# Optimal ağırlıklar pie chart
w_vals   = [w_agg_opt, w_model_opt, w_sem_opt]
w_labels = ['Aggregate', f'{best_tuned_name}', 'Semantic']
w_colors = ['steelblue', 'crimson', 'seagreen']
non_zero = [(v, l, c) for v, l, c in zip(w_vals, w_labels, w_colors) if v > 0.01]
axes[1].pie([v for v,_,_ in non_zero], labels=[l for _,l,_ in non_zero],
            colors=[c for _,_,c in non_zero], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Optimal Ensemble Ağırlıkları', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ensemble_weights.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Hata Analizi

Modelin en çok yanıldığı concern'ler ve skin type'lar hangileri?

In [ ]:
# En son fold'un val seti üzerinde hata analizi
last_train_idx, last_val_idx = cv_splits[-1]
val_df = ml_df.loc[last_val_idx].copy()

if final_model_type == 'lgbm':
    val_df['pred_score'] = final_model.predict(val_df[FEATURE_COLS_EXTENDED].values)
elif final_model_type == 'xgb':
    val_df['pred_score'] = final_model.predict(val_df[FEATURE_COLS_EXTENDED].values)
else:
    val_df['pred_score'] = final_model.predict(val_df[FEATURE_COLS_EXTENDED].values)

# Concern bazında NDCG@10
concern_ndcg = []
for concern, grp_concern in val_df.groupby('concern'):
    c_ndcgs = []
    for qid, grp in grp_concern.groupby('query_id_enc'):
        if len(grp) >= 5:
            try:
                n = ndcg_score(grp['relevance_label'].values.reshape(1,-1),
                               grp['pred_score'].values.reshape(1,-1), k=10)
                c_ndcgs.append(n)
            except Exception:
                pass
    if c_ndcgs:
        concern_ndcg.append({'concern': concern, 'ndcg10': np.mean(c_ndcgs), 'n_queries': len(c_ndcgs)})

concern_ndcg_df = pd.DataFrame(concern_ndcg).sort_values('ndcg10', ascending=False)
print('=== CONCERN BAZINDA NDCG@10 ===')
print(concern_ndcg_df.to_string(index=False))

In [ ]:
# Skin type bazında NDCG@10
skin_ndcg = []
for skin, grp_skin in val_df.groupby('skin_type'):
    s_ndcgs = []
    for qid, grp in grp_skin.groupby('query_id_enc'):
        if len(grp) >= 5:
            try:
                n = ndcg_score(grp['relevance_label'].values.reshape(1,-1),
                               grp['pred_score'].values.reshape(1,-1), k=10)
                s_ndcgs.append(n)
            except Exception:
                pass
    if s_ndcgs:
        skin_ndcg.append({'skin_type': skin, 'ndcg10': np.mean(s_ndcgs), 'n_queries': len(s_ndcgs)})

skin_ndcg_df = pd.DataFrame(skin_ndcg).sort_values('ndcg10', ascending=False)
print('=== SKIN TYPE BAZINDA NDCG@10 ===')
print(skin_ndcg_df.to_string(index=False))

In [ ]:
# Heatmap: concern × skin_type performansı
heat_data = []
for concern, grp_c in val_df.groupby('concern'):
    for skin, grp_cs in grp_c.groupby('skin_type'):
        for qid, grp in grp_cs.groupby('query_id_enc'):
            if len(grp) >= 5:
                try:
                    n = ndcg_score(grp['relevance_label'].values.reshape(1,-1),
                                   grp['pred_score'].values.reshape(1,-1), k=10)
                    heat_data.append({'concern': concern, 'skin_type': skin, 'ndcg10': n})
                except Exception:
                    pass

heat_df = pd.DataFrame(heat_data)
if not heat_df.empty:
    pivot = heat_df.groupby(['concern', 'skin_type'])['ndcg10'].mean().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title('NDCG@10 Heatmap — Concern × Skin Type', fontsize=14, fontweight='bold')
    ax.set_xlabel('Skin Type')
    ax.set_ylabel('Concern')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'performance_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Heatmap kaydedildi.')
else:
    print('Heatmap için yeterli veri yok.')

In [ ]:
# En çok yanılan query'ler (en düşük NDCG)
error_analysis = []
for (concern, skin), grp in val_df.groupby(['concern', 'skin_type']):
    for qid, q_grp in grp.groupby('query_id_enc'):
        if len(q_grp) >= 5:
            try:
                n = ndcg_score(q_grp['relevance_label'].values.reshape(1,-1),
                               q_grp['pred_score'].values.reshape(1,-1), k=10)
                error_analysis.append({
                    'concern': concern, 'skin_type': skin,
                    'ndcg10': n, 'n_products': len(q_grp)
                })
            except Exception:
                pass

error_df = pd.DataFrame(error_analysis).sort_values('ndcg10')
print('=== EN DÜŞÜK PERFORMANSLI QUERY\'LER ===')
print(error_df.head(10).to_string(index=False))
print('\n=== EN YÜKSEK PERFORMANSLI QUERY\'LER ===')
print(error_df.tail(10).sort_values('ndcg10', ascending=False).to_string(index=False))

## 10. Model Kaydetme

In [ ]:
# Final model kaydet
if final_model_type == 'lgbm':
    final_model.save_model(str(MODEL_DIR / 'final_ranker.txt'))
elif final_model_type == 'xgb':
    final_model.save_model(str(MODEL_DIR / 'final_ranker.json'))
else:
    final_model.save_model(str(MODEL_DIR / 'final_ranker_catboost'))

# ML scoring tablosu
ml_df.to_parquet(MODEL_DIR / 'ml_scoring_table.parquet', index=False)

# Konfigürasyon
final_config = {
    'best_model': best_tuned_name,
    'model_type': final_model_type,
    'feature_cols': FEATURE_COLS_EXTENDED,
    'sbert_model': 'all-MiniLM-L6-v2',
    'optimal_weights': {
        'w_aggregate': round(w_agg_opt, 4),
        'w_model': round(w_model_opt, 4),
        'w_semantic': round(w_sem_opt, 4),
    },
    'cv_results': {
        k: {'ndcg5': round(r.mean_ndcg5, 4), 'ndcg10': round(r.mean_ndcg10, 4)}
        for k, r in {**all_results, **tuned_results}.items()
    },
    'best_model_ndcg10': round(tuned_results[best_tuned_key].mean_ndcg10, 4),
    'optuna_trials': N_TRIALS,
    'effect_weights': EFFECT_WEIGHTS,
    'default_min_reviews': 3,
}

with open(MODEL_DIR / 'config.json', 'w') as f:
    json.dump(final_config, f, indent=2)

# Label encoders
with open(MODEL_DIR / 'label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

# Optuna study'leri kaydet
with open(OUTPUT_DIR / 'optuna_studies.pkl', 'wb') as f:
    pickle.dump({
        'lgbm': lgbm_study, 'xgb': xgb_study,
        'catboost': cb_study, 'weights': weight_study
    }, f)

print('\n✅ Tüm model artefactları kaydedildi:')
print(f'  {MODEL_DIR}/final_ranker.*')
print(f'  {MODEL_DIR}/ml_scoring_table.parquet')
print(f'  {MODEL_DIR}/config.json')
print(f'  {MODEL_DIR}/label_encoders.pkl')
print(f'  {OUTPUT_DIR}/optuna_studies.pkl')

In [ ]:
# Özet rapor
print('\n' + '='*70)
print('  EXPERIMENT ÖZET RAPORU')
print('='*70)
print(f'\n  Tarih     : {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}')
print(f'  CV Strateji: {N_SPLITS}-Fold Group K-Fold (concern × skin_type)')
print(f'  Feature    : Extended ({len(FEATURE_COLS_EXTENDED)} adet)')
print()
print('  BASELINE SONUÇLAR (NDCG@10):')
for k, r in all_results.items():
    print(f'    {r.model_name:<30} {r.mean_ndcg10:.4f} ± {r.std_ndcg10:.4f}')
print()
print('  TUNED SONUÇLAR (NDCG@10):')
for k, r in tuned_results.items():
    gain = r.mean_ndcg10 - all_results[k.replace('_tuned', '') if k.replace('_tuned', '') in all_results else 'lgbm'].mean_ndcg10
    print(f'    {r.model_name:<30} {r.mean_ndcg10:.4f} ± {r.std_ndcg10:.4f}  (Δ{gain:+.4f})')
print()
print(f'  🏆 FINAL MODEL : {best_tuned_name}')
print(f'     NDCG@10    : {tuned_results[best_tuned_key].mean_ndcg10:.4f}')
print(f'     Optuna trials (toplam): {N_TRIALS * 3}')
print()
print('  OPTIMAL ENSEMBLE AĞIRLIKLARI:')
print(f'    w_aggregate : {w_agg_opt:.4f}')
print(f'    w_model     : {w_model_opt:.4f}')
print(f'    w_semantic  : {w_sem_opt:.4f}')
print(f'    Ensemble NDCG@10: {weight_study.best_value:.4f}')
print('='*70)